In [ ]:
from pyspark.errors import PySparkException
from py4j.protocol import Py4JJavaError
from pyspark.sql.functions import current_timestamp
from pyspark.sql import functions as F

In [ ]:
def consulta_tabela_jdbc(jdbc_url, propriedades_conexao, nome_tabela_completa, numero_particao, coluna_referencia_paralelizacao = None):
    """Realiza a leitura de uma tabela via JDBC, permitindo a paralelização da carga.

    Args:
        jdbc_url (str): URL de conexão com o banco de dados.
        propriedades_conexao (dict): Dicionário com credenciais e driver de conexão.
        nome_tabela_completa (str): Nome da tabela de origem (ex: 'dbo.clientes' ou subquery).
        numero_particao (int/str): Quantidade de partições que o Spark usará para ler os dados.
        coluna_referencia_paralelizacao (str, optional): Coluna numérica (id) ou de data usada para
            dividir a carga entre as partições. Padrão é None (leitura sequencial).

    Returns:
        DataFrame: Spark DataFrame contendo os dados resultantes da consulta.
    """
    # Se uma coluna de partição for informada, calcula os limites (MIN/MAX) para paralelizar a leitura
    if coluna_referencia_paralelizacao is not None:
        limites_df = spark.read.jdbc(
            url = jdbc_url,
            table = f"(SELECT MIN({coluna_referencia_paralelizacao}) AS min_atributo, MAX({coluna_referencia_paralelizacao}) AS max_atributo FROM {nome_tabela_completa}) AS bounds",
            properties = propriedades_conexao
        )

        limites = limites_df.collect()[0]
        limite_inferior = limites['min_atributo']
        limite_superior = limites['max_atributo']

        df_temp = spark.read.jdbc(
            url = jdbc_url, 
            table = nome_tabela_completa,
            column = coluna_referencia_paralelizacao, # Coluna usada para paralelização
            lowerBound = limite_inferior if limite_inferior is not None else None, # Limite inferior
            upperBound = limite_superior if limite_superior is not None else None, # Limite superior 
            numPartitions = int(numero_particao), # Número de partições usada na paralelização
            properties = propriedades_conexao
        )
    else:
        df_temp = spark.read.jdbc(
            url = jdbc_url, 
            table = nome_tabela_completa,
            properties = propriedades_conexao
        )
    
    return df_temp

In [ ]:
def ingestao_incremental_hash(jdbc_url, propriedades_conexao, nome_tabela, nome_catalog, numero_particao, coluna_referencia_paralelizacao = None):
    """Gerencia o pipeline de ingestão de dados combinando JDBC e a estratégia de Hashing (MD5).

    Esta função automatiza a captura de alterações (CDC lógico) na camada de staging. 
    Ela calcula dinamicamente uma "impressão digital" (_hash) para cada linha da tabela de 
    origem. Em cargas subsequentes, em vez de comparar a tabela inteira coluna por coluna, o 
    Spark realiza um join otimizado (left_anti) trazendo apenas a coluna de hash da última versão 
    do Delta Lake (Time Travel), isolando de forma performática registros novos e modificados.

    Args:
        jdbc_url (str): URL de conexão JDBC formatada para o banco de dados de origem.
        propriedades_conexao (dict): Dicionário contendo as credenciais de autenticação e especificações do driver.
        nome_tabela (str): Nome simples da tabela no banco de dados relacional (ex: 'clientes').
        nome_catalog (str): Caminho do catálogo e schema de destino no Unity Catalog.
        numero_particao (int or str): Quantidade de partições que o Spark usará para fatiar e paralelizar a leitura dos dados via JDBC.
        coluna_referencia_paralelizacao (str, optional): Nome da coluna numérica (ex: 'id') ou de data 
            (ex: 'updated_at') utilizada para calcular os limites (lower/upper bounds) da query JDBC. 
            Se omitido (None), a leitura será estritamente sequencial (uma única thread).

    Returns:
        tuple: Uma tupla contendo dois elementos (DataFrame, str) ou (None, None):
            - elemento 0 (DataFrame or None): Spark DataFrame contendo apenas as linhas novas ou 
              alteradas, já acrescido das colunas de auditoria (`_ingested_at`, `_action`, `_hash`). 
              Retorna `None` se nenhuma alteração for detectada.
            - elemento 1 (str or None): Sugestão do modo de gravação para o Delta Lake 
              (`"overwrite"` para cargas iniciais/tabelas legadas, `"append"` para deltas incrementais, 
              ou `None` se não houver alterações).
    """
    nome_tabela_origem = f"dbo.{nome_tabela}"
    nome_tabela_destino = f"{nome_catalog}.{nome_tabela}"

    # 1. Busca os dados atuais da origem via JDBC
    df_temp = consulta_tabela_jdbc(jdbc_url, propriedades_conexao, nome_tabela_origem, numero_particao, coluna_referencia_paralelizacao)
    
    # 2. Cria a coluna de Hash dinamicamente combinando TODAS as colunas originais da tabela.
    # Tratamos valores nulos com coalesce para garantir que IDs nulos ou strings vazias não colidam hashes.
    colunas_originais = df_temp.columns
    df_origem_com_hash = df_temp.withColumn(
        "_hash", F.md5(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in colunas_originais]))
    )

    # Quantidade de linhas na tabela de origem
    qtd_linhas_origem = df_temp.count()
    print(f"[{nome_tabela}] A quantidade de linhas na tabela origem (SLQ Server): {qtd_linhas_origem}")
    

    # Cenário 1: Tabela não existe no destino -> Carga Inicial
    if not spark.catalog.tableExists(nome_tabela_destino):
        print(f"[{nome_tabela}] Primeira ingestão (Carga Inicial com Hash) da tabela: {nome_tabela}")
        df_temp_com_data_acao = df_origem_com_hash \
            .withColumn("_ingested_at", F.date_format(F.current_timestamp(), "yyyy-MM-dd HH:mm:ss")) \
            .withColumn("_action", F.lit("CARGA_INICIAL"))
        
        return df_temp_com_data_acao, "overwrite"
            
    # Cenário 2: Tabela já existe -> Filtra deltas usando comparação rápida de Hash
    else:
        # Pega as colunas atuais da tabela destino antes de ler
        colunas_destino = spark.table(nome_tabela_destino).columns
        
        # Se a tabela existe mas NÃO tem a coluna '_hash' (legado), força o comportamento de Carga Inicial
        if "_hash" not in colunas_destino:
            print(f"[{nome_tabela}] A tabela {nome_tabela} existe, mas não possui a coluna '_hash'. Forçando carga inicial para atualizar schema...")
            df_temp_com_data_acao = df_origem_com_hash \
                .withColumn("_ingested_at", F.date_format(F.current_timestamp(), "yyyy-MM-dd HH:mm:ss")) \
                .withColumn("_action", F.lit("CARGA_INICIAL"))
            
            print(f"[{nome_tabela}] Primeira ingestão (Carga Inicial com Hash) da tabela: {nome_tabela}")
            return df_temp_com_data_acao, "overwrite"
            
        # Obtém o histórico mais recente do Delta Lake para evitar leituras de snapshots fantasmas
        ultima_versao = spark.sql(f"DESCRIBE HISTORY {nome_tabela_destino}").first()["version"]
        
        # Lê o destino trazendo APENAS a coluna de hash para a memória (I/O mínimo)
        print(f"[{nome_tabela}] Verificando se há alterações entre as tabelas")
        df_destino_hashes = spark.read.format("delta") \
            .option("versionAsOf", ultima_versao) \
            .table(nome_tabela_destino) \
            .select("_hash")
                
        # O left_anti join retorna tudo o que está na origem mas NÃO foi encontrado no destino
        df_deltas = df_origem_com_hash.join(df_destino_hashes, on="_hash", how="left_anti")
        numero_alteracoes = df_deltas.count()
        
        if numero_alteracoes > 0:
            df_temp_com_data_acao = df_deltas \
                .withColumn("_ingested_at", F.date_format(F.current_timestamp(), "yyyy-MM-dd HH:mm:ss")) \
                .withColumn("_action", F.lit("INSERT_OU_UPDATE"))
            
            print(f"[{nome_tabela}] Foram identificados {numero_alteracoes} registros novos/alterados na tabela {nome_tabela}")
            return df_temp_com_data_acao, "append"
        else:
            print(f"[{nome_tabela}] Nenhuma alteração detectada para a tabela {nome_tabela}.")
            return None, None

In [ ]:
# Recupera as credenciais de acesso ao banco de dados utilizando o Databricks Secrets
usuario = dbutils.secrets.get("databases_lh_nautical", "sql_server_user")
senha = dbutils.secrets.get("databases_lh_nautical", "sql_server_password")
host = dbutils.secrets.get("databases_lh_nautical", "sql_server_host")
port = dbutils.secrets.get("databases_lh_nautical", "sql_server_port")

In [ ]:
# Montagem da String de Conexão JDBC para SQL Server
jdbc_url = f"jdbc:sqlserver://{host}:{port};databaseName=lh_nautical;encrypt=false;trustServerCertificate=true;"
propriedades_conexao = {
    "user": usuario,
    "password": senha,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Coleta de parâmetros operacionais definidos no bundle
nome_catalog = dbutils.widgets.get("catalog_name")
schema = dbutils.widgets.get("schema")
numero_particao = dbutils.widgets.get("numPartitions")

In [ ]:
# Consulta dinâmica no Information Schema para identificar todas as Tabelas Base do schema 'dbo'
query_tabelas = "(SELECT table_name FROM information_schema.tables WHERE table_schema = 'dbo' AND table_type = 'Base table') AS tabelas"

try:
    # Executa a busca e extrai os nomes das tabelas para uma lista Python nativa
    df_tabelas = consulta_tabela_jdbc(jdbc_url, propriedades_conexao, query_tabelas, numero_particao)
    lista_tabelas = [row['table_name'] for row in df_tabelas.collect()]

except Py4JJavaError as e:
    erro_msg = str(e)
    print("[Erro de conexão/JDBC] - Falha ao conectar ou executar a query no banco de dados.")
    print(f"Detalhes do erro: {erro_msg}")
    lista_tabelas = []
    raise e
  
except PySparkException as e:
    erro_msg = str(e)
    print(f"[Erro do Spark] - Ocorreu uma falha interna no Spark: {erro_msg}")
    lista_tabelas = []
    raise e

except Exception as e:
    print(f"[Erro Inesperado] - {str(e)}")
    lista_tabelas = []
    raise e

In [ ]:
# Ingestão de dados
tabelas_com_erro = []
nome_catalog_completo = f"{nome_catalog}.{schema}"

# Iteração sobre a lista de tabelas para processamento e salvamento
for nome_tabela in lista_tabelas:
    print(f"\nImportanto a tabela: {nome_tabela}...")
    nome_tabela_destino = f"{nome_catalog_completo}.{nome_tabela}"
    
    try:
        # Regra de Negócio: Define a coluna de paralelização com base nas características da tabela
        if nome_tabela not in ['variant_attribute_values', 'product_suppliers', 'stock_levels']:
            # Tabelas padrão utilizam a chave primária incremental 'id'
            df_ingestao, modo_escrita = ingestao_incremental_hash(jdbc_url, propriedades_conexao, nome_tabela, nome_catalog_completo, numero_particao, "id")

        elif nome_tabela in ['product_suppliers', 'stock_levels']:
            # Tabelas de transição utilizam carimbo de data 'updated_at'
            df_ingestao, modo_escrita = ingestao_incremental_hash(jdbc_url, propriedades_conexao, nome_tabela, nome_catalog_completo, numero_particao, "updated_at")

        else:
            # Caso caia em alguma exceção não mapeada, executa sem chave de partição (sequencial)
            df_ingestao, modo_escrita = ingestao_incremental_hash(jdbc_url, propriedades_conexao, nome_tabela, nome_catalog_completo, numero_particao)

        # Ingestão das tabelas com proteção dinâmica contra duplicações
        if df_ingestao is not None:
            # aplica 'overwrite' para não duplicar dados. Caso contrário, faz 'append'.
            if modo_escrita == "overwrite" and spark.catalog.tableExists(nome_tabela_destino):
                print(f"[{nome_tabela}] Tabela legada detectada. Aplicando OVERWRITE para reformatação segura de schema.")


            df_ingestao.write.format("delta") \
                .mode(modo_escrita) \
                .option("mergeSchema", "true") \
                .saveAsTable(nome_tabela_destino)
            
        qtd_linhas_destino = spark.table(f"{nome_tabela_destino}").count()
        print(f"[{nome_tabela}] Quantidade de Linhas no Delta: {qtd_linhas_destino}")
            
    except Py4JJavaError as e:
        erro_msg = str(e)
        print("[Erro de conexão/JDBC] - Falha ao conectar ou executar a query no banco de dados." )
        print(f"Erro ao processar tabela {nome_tabela}: {erro_msg}")
        # Registra a tabela com falha no relatório final e propaga o erro para alertar a orquestração
        tabelas_com_erro.append(nome_tabela)
        continue
    
    except PySparkException as e:
        erro_msg = str(e)
        print(f"[Erro do Spark] - Ocorreu uma falha interna no Spark: {erro_msg}")
        print(f"Erro ao processar tabela {nome_tabela}: {erro_msg}")
        # Registra a tabela que falhou para o resumo final
        tabelas_com_erro.append(nome_tabela)
        # Dispara o e-mail de alerta
        continue
    
    except Exception as e:
        erro_msg = str(e)
        print(f"[Erro Inesperado] - {erro_msg}")
        # Registra a tabela que falhou para o resumo final
        tabelas_com_erro.append(nome_tabela)
        # Dispara o e-mail de alerta
        continue

# Resumo da execução do Pipeline
print("\nIngestão finalizada!")
if tabelas_com_erro:
    print(f"Atenção! As seguintes tabelas falharam e geraram alertas: {tabelas_com_erro}")
    raise Exception(f"Falha na ingestão das seguintes tabelas: {tabelas_com_erro}")
else:
    print("Sucesso absoluto! Todas as tabelas foram copiadas sem erros.")